In [1]:
# This notebook answers RQ3: Which cantons offer the best opportunities for combining different outdoor activities (hiking, cycling, mountain biking, skating)?

import pandas as pd
# pandas: works with tables (DataFrames).
# Used to load the routes, count them per canton and activity, and calculate percentile ranks.

In [2]:
# The four route tables with cantons (result of canton_assignment.ipynb) are stored in a list.
# The four route tables with cantons (result of canton_assignment.ipynb)
files = [
    "result_wanderland_cantons.csv",
    "result_veloland_cantons.csv",
    "result_mountainbikeland_cantons.csv",
    "result_skatingland_cantons.csv",
]

In [3]:
routes = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)

In [5]:
# Check: number of routes per activity and in total
print(routes["land"].value_counts())
print("Total:", len(routes))

land
mountainbikeland    286
wanderland          274
veloland             65
skatingland          11
Name: count, dtype: int64
Total: 636


In [6]:
# One row per route and canton (a route counts for every canton in which it starts or ends),
# together with the activity type of the route.
route_canton = pd.concat([
    routes[["start_canton", "land"]].rename(columns={"start_canton": "canton"}),
    routes[["end_canton", "land"]].rename(columns={"end_canton": "canton"}),
])

# A route that starts and ends in the same canton counts only once for that canton
route_canton = route_canton.reset_index().drop_duplicates()

# Routes abroad are not counted
route_canton = route_canton[route_canton["canton"] != "abroad"]

# Check: number of route-canton combinations
print("Route-canton combinations:", len(route_canton))

Route-canton combinations: 648


In [7]:
# Number of routes per canton (rows) and activity (columns)
routes_per_activity = pd.crosstab(route_canton["canton"], route_canton["land"])

print(routes_per_activity.to_string())

land    mountainbikeland  skatingland  veloland  wanderland
canton                                                     
AG                     1            0         1           0
AI                     1            0         1           2
AR                     0            0         1           5
BE                    19            1        15          35
BL                     1            0         5           4
BS                     0            1         1           0
FR                     0            2         1          15
GE                     0            0         3           6
GL                     5            0         0          11
GR                    97            1         3          50
JU                    10            0         1           7
LU                     1            0         2           2
NE                     9            1         3           4
NW                     6            0         0          10
OW                     5            0   

In [8]:
# Load the geographic key figures of the cantons (BFS, as of 01.01.2026)
canton_figures = pd.read_excel("Geographische_Kennzahlen.xlsx", sheet_name="Daten")

# Canton abbreviation for each official canton number (order of the Federal Constitution)
canton_abbreviations = {
    1: "ZH", 2: "BE", 3: "LU", 4: "UR", 5: "SZ", 6: "OW", 7: "NW", 8: "GL", 9: "ZG",
    10: "FR", 11: "SO", 12: "BS", 13: "BL", 14: "SH", 15: "AR", 16: "AI", 17: "SG",
    18: "GR", 19: "AG", 20: "TG", 21: "TI", 22: "VD", 23: "VS", 24: "NE", 25: "GE", 26: "JU"
}
canton_figures["canton"] = canton_figures["KTNR"].map(canton_abbreviations)

# Convert the area from hectares to square kilometres (1 km² = 100 ha)
canton_figures["area_km2"] = canton_figures["AREA_HA"] / 100

# Check: number of cantons
print("Cantons:", len(canton_figures))

Cantons: 26


In [9]:
# Include all 26 cantons: cantons without any route (Zug) get 0 for every activity
routes_per_activity = routes_per_activity.reindex(canton_figures["canton"], fill_value=0)

# Area of each canton, with the canton abbreviation as row name
area = canton_figures.set_index("canton")["area_km2"]

# Density: number of routes per 100 km², calculated separately for each activity
density = routes_per_activity.div(area, axis=0) * 100

print(density.round(2).to_string())

land    mountainbikeland  skatingland  veloland  wanderland
canton                                                     
ZH                  0.00         0.12      0.12        0.29
BE                  0.32         0.02      0.25        0.59
LU                  0.07         0.00      0.13        0.13
UR                  1.76         0.00      0.00        0.56
SZ                  1.10         0.00      0.00        0.99
OW                  1.02         0.00      0.61        1.22
NW                  2.18         0.00      0.00        3.63
GL                  0.73         0.00      0.00        1.61
ZG                  0.00         0.00      0.00        0.00
FR                  0.00         0.12      0.06        0.90
SO                  0.25         0.25      0.38        0.76
BS                  0.00         2.71      2.71        0.00
BL                  0.19         0.00      0.97        0.77
SH                  0.00         0.34      1.68        1.68
AR                  0.00         0.00   

In [10]:
# Convert each density into a percentile rank across all 26 cantons.
# Each activity (column) is ranked separately: 1 = highest density of all cantons for this activity.
# Cantons with the same density share the same (average) rank.
ranks = density.rank(pct=True)

print(ranks.round(2).to_string())

land    mountainbikeland  skatingland  veloland  wanderland
canton                                                     
ZH                  0.17         0.81      0.42        0.19
BE                  0.50         0.69      0.58        0.38
LU                  0.35         0.33      0.50        0.15
UR                  0.96         0.33      0.12        0.31
SZ                  0.81         0.33      0.12        0.73
OW                  0.77         0.33      0.85        0.81
NW                  1.00         0.33      0.12        1.00
GL                  0.62         0.33      0.12        0.85
ZG                  0.17         0.33      0.12        0.08
FR                  0.17         0.85      0.35        0.69
SO                  0.46         0.92      0.73        0.54
BS                  0.17         1.00      1.00        0.08
BL                  0.42         0.33      0.88        0.58
SH                  0.17         0.96      0.96        0.88
AR                  0.17         0.33   

In [11]:
# Combined index: mean of the four percentile ranks per canton.
# A canton only reaches a high index if it has a high density in several activities.
rq3 = ranks.copy()
rq3["combined_index"] = ranks.mean(axis=1).round(3)

# Add the total number of routes per canton, to see on how many routes the index is based
rq3["routes_total"] = routes_per_activity.sum(axis=1)

# Sort from the highest to the lowest index
rq3 = rq3.sort_values("combined_index", ascending=False)

print(rq3.round(2).to_string())

land    mountainbikeland  skatingland  veloland  wanderland  combined_index  routes_total
canton                                                                                   
SH                  0.17         0.96      0.96        0.88            0.74            11
OW                  0.77         0.33      0.85        0.81            0.69            14
NE                  0.85         0.88      0.69        0.27            0.67            17
SO                  0.46         0.92      0.73        0.54            0.66            13
AI                  0.58         0.33      0.81        0.77            0.62             4
NW                  1.00         0.33      0.12        1.00            0.61            16
SG                  0.65         0.73      0.54        0.46            0.60            36
GE                  0.17         0.33      0.92        0.96            0.60             9
GR                  0.92         0.65      0.27        0.50            0.59           151
JU        

In [12]:
# Save the result of RQ3: percentile ranks per activity and combined index per canton
rq3.to_csv("rq3_result_combined_index.csv")
print("Saved:", len(rq3), "cantons")

Saved: 26 cantons
